# Check Counterfactual Data

In [1]:
import pandas as pd
from transformer_lens import HookedTransformer
import re

/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B", device="cuda")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


In [7]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1cukGqysonkhQFD8_0rWgLYpMg95EPqm_5B3svOryf0A'  # Replace with your actual Google Sheet ID
gid_eng = '257590483'  # Replace with the actual GID for the English sheet
gid_indo = '257590483'  # Replace with the actual GID for the Indonesian sheet
gid_sunda = '257590483'  # Replace with the actual GID for the Sunda sheet
filled_counterfacts = {}
try:
	for lang in ['eng', 'indo', 'sunda']:
		filled_counterfacts[lang] = get_google_sheet(google_sheet_id, globals()[f'gid_{lang}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

Successfully loaded data from the specific sheet:


In [8]:
for lang in ['eng', 'indo', 'sunda']:
	if lang != 'indo':
		continue
	filled_counterfacts[lang] = pd.read_csv(f'hotel_dataset/counterfactsv2.1/clean_traintruncated/{lang}_counterfacts.csv')
filled_counterfacts['indo']

,index,original_pair,corrupted_pair
0,2501,pelayanan lumayan baik . [A] [O] [S] [A] pelay...,satelit sangat wangi . [A] [O] [S] [A] satelit...
1,2506,suasananya kurang nyaman . untuk menginap haru...,keretanya cukup ramah . untuk menginap harus d...
2,2512,makanan yang kurang memuaskan . [A] [O] [S] [A...,sepeda yang berwangi bunga . [A] [O] [S] [A] s...
3,2516,tempatnya nyaman untuk istirahat . [A] [O] [S]...,meja nya ribut untuk istirahat . [A] [O] [S] [...
4,2517,kamar bersih . [A] [O] [S] [A] kamar [O] bersi...,daun bau . [A] [O] [S] [A] daun [O] bau [S] ne...
...,...,...,...
65,2618,hotelnya bagus bersih nyaman . [A] [O] [S] [A]...,kapal galak malu kasar . [A] [O] [S] [A] kapal...
66,2696,tempatnya bersih dan nyaman . staf nya ramah ....,kantong gelap dan curang . tombolnya lama . [A...
67,2994,kamar bagus dan bersih tetapi kurang besar . [...,buku galak dan judes tetapi sangat kuat . [A] ...
68,3015,"staf kurang ramah , sarung bantal kotor , pint...","penghapus manis sekali , papan tulis gagah , t..."


In [9]:
def extract_triplet_fixed(text):
	try:
		matches = list(re.finditer(r"\[([AOS])\]", text))
		if len(matches) >= 3:
			a_start = matches[0].end()
			o_start = matches[1].end()
			s_start = matches[2].end()
			aspect = text[a_start:matches[1].start()].strip()
			opinion = text[o_start:matches[2].start()].strip()
			sentiment = text[s_start:].split()[0].strip()
			return (aspect, opinion, sentiment)
	except:
		return None

In [10]:
filled_counterfacts['indo']

,index,original_pair,corrupted_pair
0,2501,pelayanan lumayan baik . [A] [O] [S] [A] pelay...,satelit sangat wangi . [A] [O] [S] [A] satelit...
1,2506,suasananya kurang nyaman . untuk menginap haru...,keretanya cukup ramah . untuk menginap harus d...
2,2512,makanan yang kurang memuaskan . [A] [O] [S] [A...,sepeda yang berwangi bunga . [A] [O] [S] [A] s...
3,2516,tempatnya nyaman untuk istirahat . [A] [O] [S]...,meja nya ribut untuk istirahat . [A] [O] [S] [...
4,2517,kamar bersih . [A] [O] [S] [A] kamar [O] bersi...,daun bau . [A] [O] [S] [A] daun [O] bau [S] ne...
...,...,...,...
65,2618,hotelnya bagus bersih nyaman . [A] [O] [S] [A]...,kapal galak malu kasar . [A] [O] [S] [A] kapal...
66,2696,tempatnya bersih dan nyaman . staf nya ramah ....,kantong gelap dan curang . tombolnya lama . [A...
67,2994,kamar bagus dan bersih tetapi kurang besar . [...,buku galak dan judes tetapi sangat kuat . [A] ...
68,3015,"staf kurang ramah , sarung bantal kotor , pint...","penghapus manis sekali , papan tulis gagah , t..."


In [11]:
lang = 'indo'  # Change this to 'indo' or 'sunda' as needed
# Original sentence
original_texts = filled_counterfacts[lang]['original_pair'].tolist()

# Candidate counterfactuals
candidate_texts = filled_counterfacts[lang]['corrupted_pair'].tolist()

indexes = filled_counterfacts[lang]['index'].tolist()

count_mismatches = 0
for idx, (original_text, candidate) in enumerate(zip(original_texts, candidate_texts)):
	original_tokens = model.to_str_tokens(original_text, prepend_bos=False)
	original_len = len(original_tokens)

	# Extract original prompt
	original_prompt = original_text.split("[A] [O] [S]")[0].strip()
	original_prompt_len = len(model.to_str_tokens(original_prompt, prepend_bos=False))

	# Extract original aspect and opinion
	original_split = original_text.split("[A] [O] [S]")[-1].strip()
	original_split = original_split.split("[SSEP]")
	original_split = [triplet.strip() for triplet in original_split]
	original_aspects, original_opinions = [], []
	original_aspect_tokens, original_opinion_tokens = [], []
	original_aspect_lens, original_opinion_lens = [], []
	for i, original_triplet in enumerate(original_split):

		original_aspect, original_opinion, _ = extract_triplet_fixed(original_triplet)
		original_aspects.append(f' {original_aspect}')
		original_opinions.append(f' {original_opinion}')
		original_aspect_tokens.append(model.to_str_tokens(f' {original_aspect}', prepend_bos=False))
		original_opinion_tokens.append(model.to_str_tokens(f' {original_opinion}', prepend_bos=False))
		original_aspect_lens.append(len(original_aspect_tokens[-1]))
		original_opinion_lens.append(len(original_opinion_tokens[-1]))

	candidate_tokens = model.to_str_tokens(candidate, prepend_bos=False)
	candidate_len = len(candidate_tokens)

	# Extract prompt
	candidate_prompt = candidate.split("[A] [O] [S]")[0].strip()
	candidate_prompt_len = len(model.to_str_tokens(candidate_prompt, prepend_bos=False))
	prompt_match = candidate_prompt_len == original_prompt_len

	# Extract candidate aspect and opinion
	candidate_split = candidate.split("[A] [O] [S]")[-1].strip()
	candidate_split = candidate_split.split("[SSEP]")
	candidate_split = [triplet.strip() for triplet in candidate_split]
	candidate_aspects, candidate_opinions = [], []
	candidate_aspect_tokens, candidate_opinion_tokens = [], []
	candidate_aspect_lens, candidate_opinion_lens = [], []
	for i, candidate_triplet in enumerate(candidate_split):
		candidate_aspect, candidate_opinion, _ = extract_triplet_fixed(candidate_triplet)
		candidate_aspects.append(f' {candidate_aspect}')
		candidate_opinions.append(f' {candidate_opinion}')
		candidate_aspect_tokens.append(model.to_str_tokens(f' {candidate_aspect}', prepend_bos=False))
		candidate_opinion_tokens.append(model.to_str_tokens(f' {candidate_opinion}', prepend_bos=False))
		candidate_aspect_lens.append(len(candidate_aspect_tokens[-1]))
		candidate_opinion_lens.append(len(candidate_opinion_tokens[-1]))

	aspect_matches = [candidate_aspect_lens[i] == original_aspect_lens[i] for i in range(len(original_aspects))]
	opinion_matches = [candidate_opinion_lens[i] == original_opinion_lens[i] for i in range(len(original_opinions))]


	if not (prompt_match and all(aspect_matches) and all(opinion_matches)):
		print(f"Original Sentence of index {indexes[idx]}:")
		print(f"Original: {original_text}")
		print(f"Candidate: {candidate}")
		print(f"Original prompt     : '{original_prompt}' → Length: {original_prompt_len}")
		print(f"Candidate prompt    : '{candidate_prompt}' → Length: {candidate_prompt_len} (match: {prompt_match})")
		for i in range(len(original_aspects)):
			aspect = original_aspects[i]
			opinion = original_opinions[i]
			aspect_tokens = original_aspect_tokens[i]
			opinion_tokens = original_opinion_tokens[i]
			aspect_len = original_aspect_lens[i]
			opinion_len = original_opinion_lens[i]

			candidate_aspect = candidate_aspects[i]
			candidate_opinion = candidate_opinions[i]
			candidate_aspect_tokens = candidate_aspect_tokens[i]
			candidate_opinion_tokens = candidate_opinion_tokens[i]
			candidate_aspect_len = candidate_aspect_lens[i]
			candidate_opinion_len = candidate_opinion_lens[i]

			aspect_match = candidate_aspect_len == aspect_len
			opinion_match = candidate_opinion_len == opinion_len

			print(f"  Aspect {i+1} : '{aspect}' → Tokens: {aspect_tokens} → Length: {aspect_len}")
			print(f"  Opinion {i+1}: '{opinion}' → Tokens: {opinion_tokens} → Length: {opinion_len}")
			print(f"  Candidate Aspect {i+1} : '{candidate_aspect}' → Tokens: {candidate_aspect_tokens} → Length: {candidate_aspect_len} (match: {aspect_match})")
			print(f"  Candidate Opinion {i+1}: '{candidate_opinion}' → Tokens: {candidate_opinion_tokens} → Length: {candidate_opinion_len} (match: {opinion_match})")
			print("-" * 80)
		print(f"Original total toks : {original_len}")
		print(f"Candidate total toks : {candidate_len}")
		print(f"Match: {prompt_match and all(aspect_matches) and all(opinion_matches)}")
		print("=" * 100)
		print("\n")
		count_mismatches += 1
	
		# # Change valid to False if any of the checks fail
		# filled_counterfacts[lang].loc[idx, 'valid'] = False

print(f"Total mismatches found: {count_mismatches}")

Total mismatches found: 0


In [ ]:
# Store to csv
filled_counterfacts['indo'].to_csv(f'temp/filled_counterfacts_indo.csv', index=False)